# Distillation Dueling Horizon Supervisor

Unified distillation horizon notebook. Edit the config cell below for one-off runs, or change `systems/distillation/notebook_params.py` for repo-wide defaults.

## User Config And Imports

In [1]:
from pathlib import Path
import os
import random

from DuelingDQN.dueling_dqn_agent import DuelingDQNAgent
from systems.distillation import get_distillation_notebook_defaults
from systems.distillation.data_io import canonical_baseline_path, load_distillation_system_data
from systems.distillation.labels import DISTILLATION_SYSTEM_METADATA
from systems.distillation.plant import build_distillation_system, distillation_system_stepper
from systems.distillation.scenarios import build_distillation_disturbance_schedule
from utils.helpers import apply_min_max, build_horizon_recipes
from utils.horizon_runner_dueling import run_dueling_dqn_mpc_horizon_supervisor
from utils.notebook_setup import prepare_distillation_notebook_env, print_grouped_notebook_summary
from utils.plotting import compare_mpc_rl_from_dirs, plot_horizon_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim

import numpy as np
import torch

NB = get_distillation_notebook_defaults("horizon_dueling")
RUN_MODE = NB["run_mode"]
DISTURBANCE_PROFILE = NB["disturbance_profile"]
STATE_MODE = NB["state_mode"]
STYLE_PROFILE = NB["style_profile"]
SAVE_PDF = NB["save_pdf"]
ASPEN_PRESET = NB["aspen_preset"]
ASPEN_PATH_OVERRIDE = NB["aspen_path_override"]
SNAPS_PATH_OVERRIDE = NB["snaps_path_override"]
ASPEN_ROOT_OVERRIDE = NB["aspen_root_override"]
DISTILLATION_VISIBLE = NB["distillation_visible"]
DISTILLATION_DATA_DIR_OVERRIDE = NB["data_dir_override"]
DISTILLATION_RESULTS_DIR_OVERRIDE = NB["results_dir_override"]
RESULT_PREFIX_OVERRIDE = NB["result_prefix_override"]
COMPARE_PREFIX_OVERRIDE = NB["compare_prefix_override"]
BASELINE_MPC_PATH_OVERRIDE = NB["baseline_mpc_path_override"]
N_TESTS_OVERRIDE = NB["n_tests_override"]
SET_POINTS_LEN_OVERRIDE = NB["set_points_len_override"]
WARM_START_OVERRIDE = NB["warm_start_override"]
TEST_CYCLE_OVERRIDE = NB["test_cycle_override"]
PLOT_START_EPISODE_OVERRIDE = NB["plot_start_episode_override"]
COMPARE_START_EPISODE_OVERRIDE = NB["compare_start_episode_override"]
REPO_ROOT, DATA_DIR, RESULT_DIR, DISTURBANCE_PROFILE, DYN_PATH, SNAPS_PATH, ASPEN_SOURCE = prepare_distillation_notebook_env(run_mode=RUN_MODE, disturbance_profile=DISTURBANCE_PROFILE, family="horizon_dueling", aspen_preset=ASPEN_PRESET, dyn_path_override=ASPEN_PATH_OVERRIDE, snaps_path_override=SNAPS_PATH_OVERRIDE, aspen_root_override=ASPEN_ROOT_OVERRIDE, data_dir_override=DISTILLATION_DATA_DIR_OVERRIDE, results_dir_override=DISTILLATION_RESULTS_DIR_OVERRIDE)
os.chdir(REPO_ROOT)


## System And Data Setup

In [2]:
# Build the plant, load the canonical data bundle, and prepare the supervisory setpoint scenario.
SYS = NB["system_setup"]
RUN_PROFILE = NB["run_profiles"][(RUN_MODE, DISTURBANCE_PROFILE)]
nominal_conditions = SYS["nominal_conditions"].copy()
ss_inputs = SYS["ss_inputs"].copy()
u_min = SYS["input_bounds"]["u_min"].copy()
u_max = SYS["input_bounds"]["u_max"].copy()
setpoint_y = SYS["setpoint_range_phys"].copy()
y_sp_scenario_phys = SYS["rl_setpoints_phys"].copy()
system = build_distillation_system(path=DYN_PATH, ss_inputs=ss_inputs, initialization_point=nominal_conditions, delta_t=SYS["delta_t_hours"], visible=DISTILLATION_VISIBLE)
steady_states = {"ss_inputs": np.asarray(system.ss_inputs, float).copy(), "y_ss": np.asarray(system.y_ss, float).copy()}
system_data = load_distillation_system_data(REPO_ROOT, steady_states, setpoint_y, u_min, u_max, data_override=DISTILLATION_DATA_DIR_OVERRIDE)
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]
inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(y_sp_scenario_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])
RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or f"distillation_dueling_horizon_{RUN_MODE}_{DISTURBANCE_PROFILE}_{STATE_MODE}_unified"
COMPARE_PREFIX = COMPARE_PREFIX_OVERRIDE or f"distillation_compare_dueling_horizon_{RUN_MODE}_{DISTURBANCE_PROFILE}_{STATE_MODE}"
BASELINE_MPC_PATH = Path(BASELINE_MPC_PATH_OVERRIDE).expanduser() if BASELINE_MPC_PATH_OVERRIDE else canonical_baseline_path(REPO_ROOT, RUN_MODE, DISTURBANCE_PROFILE, data_override=DISTILLATION_DATA_DIR_OVERRIDE)
EPISODE_CFG = NB["episode_defaults"]
n_tests = int(RUN_PROFILE.get("n_tests", EPISODE_CFG["n_tests"]) if N_TESTS_OVERRIDE is None else N_TESTS_OVERRIDE)
set_points_len = int(RUN_PROFILE.get("set_points_len", EPISODE_CFG["set_points_len"]) if SET_POINTS_LEN_OVERRIDE is None else SET_POINTS_LEN_OVERRIDE)
warm_start = int(RUN_PROFILE.get("warm_start", EPISODE_CFG["warm_start"]) if WARM_START_OVERRIDE is None else WARM_START_OVERRIDE)
TEST_CYCLE = list(RUN_PROFILE.get("test_cycle", EPISODE_CFG["test_cycle"]) if TEST_CYCLE_OVERRIDE is None else TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = int(RUN_PROFILE.get("plot_start_episode", 1) if PLOT_START_EPISODE_OVERRIDE is None else PLOT_START_EPISODE_OVERRIDE)
COMPARE_START_EPISODE = int(RUN_PROFILE.get("compare_start_episode", PLOT_START_EPISODE) if COMPARE_START_EPISODE_OVERRIDE is None else COMPARE_START_EPISODE_OVERRIDE)
TOTAL_STEPS = n_tests * set_points_len * len(y_sp_scenario_phys)
DISTURBANCE_NOMINAL_FEED = float(system.feed.FmR.Value)
DISTURBANCE_SCHEDULE = build_distillation_disturbance_schedule(RUN_MODE, DISTURBANCE_PROFILE, TOTAL_STEPS, nominal_feed=DISTURBANCE_NOMINAL_FEED)


## Run / Reward / Agent Setup

In [3]:
# Run-profile, controller, reward, and agent setup.
CTRL = NB["controller"]
AGENT_CFG = NB["agent"]
REWARD_CFG = NB["reward"]
poles = SYS["observer_poles"].copy()
PREDICT_GRID = list(CTRL["predict_grid"])
CONTROL_GRID = list(CTRL["control_grid"])
HORIZON_RECIPES = build_horizon_recipes(PREDICT_GRID, CONTROL_GRID)
STATE_DIM = get_rl_state_dim(A_aug.shape[0], C_aug.shape[0], B_aug.shape[1], STATE_MODE)
MISMATCH_CLIP = CTRL["mismatch_clip"]
INNOVATION_SCALE_MODE = CTRL["innovation_scale_mode"]
INNOVATION_SCALE_REF = CTRL["innovation_scale_ref"]
TRACKING_SCALE_MODE = CTRL["tracking_scale_mode"]
TRACKING_ETA_TOL = CTRL["tracking_eta_tol"]
TRACKING_SCALE_FLOOR = CTRL["tracking_scale_floor"]
TRACKING_SCALE_FLOOR_MODE = CTRL["tracking_scale_floor_mode"]
BASE_STATE_NORM_MODE = CTRL["base_state_norm_mode"]
BASE_STATE_RUNNING_NORM_CLIP = CTRL["base_state_running_norm_clip"]
BASE_STATE_RUNNING_NORM_EPS = CTRL["base_state_running_norm_eps"]
MISMATCH_FEATURE_TRANSFORM_MODE = CTRL["mismatch_feature_transform_mode"]
MISMATCH_TRANSFORM_TANH_SCALE = CTRL["mismatch_transform_tanh_scale"]
MISMATCH_TRANSFORM_POST_CLIP = CTRL["mismatch_transform_post_clip"]
OBSERVER_UPDATE_ALIGNMENT = CTRL["observer_update_alignment"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = int(AGENT_CFG.get("seed", 0))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
BUFFER_SIZE = int(AGENT_CFG["buffer_size"])
REPLAY_FRAC_PER = float(AGENT_CFG["replay_frac_per"])
REPLAY_FRAC_RECENT = float(AGENT_CFG["replay_frac_recent"])
REPLAY_RECENT_WINDOW_MULT = int(AGENT_CFG["replay_recent_window_mult"])
REPLAY_RECENT_WINDOW = int(AGENT_CFG["replay_recent_window"]) if AGENT_CFG["replay_recent_window"] is not None else min(BUFFER_SIZE, REPLAY_RECENT_WINDOW_MULT * set_points_len)
REPLAY_ALPHA = float(AGENT_CFG["replay_alpha"])
REPLAY_BETA_START = float(AGENT_CFG["replay_beta_start"])
REPLAY_BETA_END = float(AGENT_CFG["replay_beta_end"])
REPLAY_BETA_STEPS = int(AGENT_CFG["replay_beta_steps"])
N_STEP = int(AGENT_CFG["n_step"])
MULTISTEP_MODE = AGENT_CFG["multistep_mode"]
LAMBDA_VALUE = float(AGENT_CFG["lambda_value"])
DECISION_INTERVAL = int(CTRL["decision_interval"])
EXPLORATION_MODE = AGENT_CFG["exploration_mode"]
LOSS_TYPE = AGENT_CFG["loss_type"]
USE_SHIFTED_MPC_WARM_START = CTRL["use_shifted_mpc_warm_start"]
REPLAY_SETTINGS = {
    "buffer_size": BUFFER_SIZE,
    "replay_frac_per": REPLAY_FRAC_PER,
    "replay_frac_recent": REPLAY_FRAC_RECENT,
    "replay_recent_window_mult": REPLAY_RECENT_WINDOW_MULT,
    "replay_recent_window": REPLAY_RECENT_WINDOW,
    "replay_alpha": REPLAY_ALPHA,
    "replay_beta_start": REPLAY_BETA_START,
    "replay_beta_end": REPLAY_BETA_END,
    "replay_beta_steps": REPLAY_BETA_STEPS,
}
predict_h = CTRL["predict_h"]
cont_h = CTRL["cont_h"]
Q1_penalty = CTRL["Q1_penalty"]
Q2_penalty = CTRL["Q2_penalty"]
R1_penalty = CTRL["R1_penalty"]
R2_penalty = CTRL["R2_penalty"]
# Reward setup.
reward_params, reward_fn = make_reward_fn_relative_QR(data_min, data_max, n_inputs=2, **REWARD_CFG)
nominal_qi = CTRL["nominal_qi"]
nominal_qs = CTRL["nominal_qs"]
nominal_hA = CTRL["nominal_ha"]
qi_change = CTRL["qi_change"]
qs_change = CTRL["qs_change"]
ha_change = CTRL["ha_change"]
# Agent setup.
dqn_agent = DuelingDQNAgent(state_dim=STATE_DIM, action_dim=len(HORIZON_RECIPES), hidden_dim=list(AGENT_CFG["hidden_layers"]), gamma=AGENT_CFG["gamma"], n_step=N_STEP, multistep_mode=MULTISTEP_MODE, lambda_value=LAMBDA_VALUE, lr=AGENT_CFG["lr"], batch_size=AGENT_CFG["batch_size"], buffer_size=BUFFER_SIZE, replay_frac_per=REPLAY_FRAC_PER, replay_frac_recent=REPLAY_FRAC_RECENT, replay_recent_window=REPLAY_RECENT_WINDOW, replay_alpha=REPLAY_ALPHA, replay_beta_start=REPLAY_BETA_START, replay_beta_end=REPLAY_BETA_END, replay_beta_steps=REPLAY_BETA_STEPS, grad_clip_norm=AGENT_CFG["grad_clip_norm"], double_dqn=AGENT_CFG["double_dqn"], target_update=AGENT_CFG["target_update"], tau=AGENT_CFG["tau"], hard_update_interval=AGENT_CFG["hard_update_interval"], activation=AGENT_CFG["activation"], use_layer_norm=AGENT_CFG["use_layer_norm"], dropout=AGENT_CFG["dropout"], device=DEVICE, exploration_mode=EXPLORATION_MODE, loss_type=LOSS_TYPE, eps_start=AGENT_CFG["eps_start"], eps_end=AGENT_CFG["eps_end"], eps_decay_rate=AGENT_CFG["eps_decay_rate"], eps_decay_mode=AGENT_CFG["eps_decay_mode"], eps_decay_steps=AGENT_CFG["eps_decay_steps"])


## Resolved Summary

In [4]:
print_grouped_notebook_summary(
    "Distillation Dueling Horizon Supervisor run summary",
    {
        "Paths": {"Repo root": REPO_ROOT, "Data dir": DATA_DIR, "Results dir": RESULT_DIR, "Aspen source": ASPEN_SOURCE, "Dyn path": DYN_PATH, "Snaps path": SNAPS_PATH, "Baseline MPC": BASELINE_MPC_PATH},
        "Run setup": {"Run mode": RUN_MODE, "Disturbance profile": DISTURBANCE_PROFILE, "State mode": STATE_MODE, "n_tests": n_tests, "set_points_len": set_points_len, "warm_start": warm_start, "test_cycle": TEST_CYCLE, "decision_interval": DECISION_INTERVAL, "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START, "seed": SEED},
        "System / controller": {"delta_t_hours": SYS["delta_t_hours"], "predict_h": predict_h, "cont_h": cont_h, "predict_grid": PREDICT_GRID, "control_grid": CONTROL_GRID, "observer_poles": poles.tolist(), "setpoints_phys": y_sp_scenario_phys.tolist()},
        "Reward": reward_params,
        "Agent": {"algorithm": "dueling_ddqn", "hidden_layers": AGENT_CFG["hidden_layers"], "buffer_size": BUFFER_SIZE, "n_step": N_STEP, "multistep_mode": MULTISTEP_MODE, "lambda_value": LAMBDA_VALUE, "exploration_mode": EXPLORATION_MODE, "loss_type": LOSS_TYPE},
        "Replay": REPLAY_SETTINGS,
        "Mismatch": {"clip": MISMATCH_CLIP, "innovation_scale_mode": INNOVATION_SCALE_MODE, "tracking_scale_mode": TRACKING_SCALE_MODE, "tracking_eta_tol": TRACKING_ETA_TOL, "tracking_scale_floor_mode": TRACKING_SCALE_FLOOR_MODE},
        "Plotting / export": {"style_profile": STYLE_PROFILE, "save_pdf": SAVE_PDF, "result_prefix": RESULT_PREFIX, "compare_prefix": COMPARE_PREFIX, "plot_start_episode": PLOT_START_EPISODE, "compare_start_episode": COMPARE_START_EPISODE},
    },
)

Distillation Dueling Horizon Supervisor run summary

[Paths]
  Repo root           : C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer
  Data dir            : C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Distillation\Data
  Results dir         : C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Distillation\Results
  Aspen source        : family-default
  Dyn path            : C:\Users\HAMEDI\Desktop\FinalDocuments\FinalDocuments\C2SplitterControlFiles\AspenFiles\dynsim\Plant\C2S_SS_simulation4.dynf
  Snaps path          : C:\Users\HAMEDI\Desktop\FinalDocuments\FinalDocuments\C2SplitterControlFiles\AspenFiles\dynsim\Plant\C2S_SS_simulation4
  Baseline MPC        : C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Distillation\Data\mpc_results_disturb_fluctuation.pickle

[Run setup]
  Run mode            : disturb
  Disturbance profile : fluctuation
  State mode          : standard
  n_tests             : 200
  set_points_len      : 200
  warm_start          : 5
  test_cycle          : [Fa

## Run

In [5]:
# Assemble the shared runner configuration and execute the rollout.
dueling_cfg = {
    "mode": RUN_MODE,
    "state_mode": STATE_MODE,
    "algorithm": "dueling_ddqn",
        "mismatch_clip": MISMATCH_CLIP,
    "innovation_scale_mode": INNOVATION_SCALE_MODE,
    "innovation_scale_ref": INNOVATION_SCALE_REF,
    "tracking_scale_mode": TRACKING_SCALE_MODE,
    "tracking_eta_tol": TRACKING_ETA_TOL,
    "tracking_scale_floor": TRACKING_SCALE_FLOOR,
    "tracking_scale_floor_mode": TRACKING_SCALE_FLOOR_MODE,
    "base_state_norm_mode": BASE_STATE_NORM_MODE,
    "base_state_running_norm_clip": BASE_STATE_RUNNING_NORM_CLIP,
    "base_state_running_norm_eps": BASE_STATE_RUNNING_NORM_EPS,
    "mismatch_feature_transform_mode": MISMATCH_FEATURE_TRANSFORM_MODE,
    "mismatch_transform_tanh_scale": MISMATCH_TRANSFORM_TANH_SCALE,
    "mismatch_transform_post_clip": MISMATCH_TRANSFORM_POST_CLIP,
    "observer_update_alignment": OBSERVER_UPDATE_ALIGNMENT,
    "notebook_source": "distillation_RL_assisted_MPC_horizons_dueling_unified.ipynb",
    "predict_h": predict_h,
    "cont_h": cont_h,
    "decision_interval": DECISION_INTERVAL,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "n_step": N_STEP,
    "multistep_mode": MULTISTEP_MODE,
    "lambda_value": LAMBDA_VALUE,
    "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
}

runtime_ctx = {
    "system": system,
    "y_sp_scenario": y_sp_scenario,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "agent": dqn_agent,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": poles,
    "data_min": data_min,
    "data_max": data_max,
    "horizon_recipes": HORIZON_RECIPES,
    "reward_fn": reward_fn,
    "system_metadata": DISTILLATION_SYSTEM_METADATA,
    "reward_params": reward_params,
    "system_stepper": distillation_system_stepper,
    "disturbance_labels": DISTILLATION_SYSTEM_METADATA.get("disturbance_labels"),
    "disturbance_schedule": DISTURBANCE_SCHEDULE,
}

result_bundle = run_dueling_dqn_mpc_horizon_supervisor(dueling_cfg=dueling_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = BASELINE_MPC_PATH
result_bundle["reward_params"] = reward_params
result_bundle["run_profile"] = RUN_PROFILE


Sub_Episode: 1 | avg. reward: 11.679969339929185 | Hp,Hc: (6, 3)
Sub_Episode: 2 | avg. reward: 17.69391483756164 | Hp,Hc: (6, 3)
Sub_Episode: 3 | avg. reward: 17.686632979595263 | Hp,Hc: (6, 3)
Sub_Episode: 4 | avg. reward: 17.682135328648418 | Hp,Hc: (6, 3)
Sub_Episode: 5 | avg. reward: 17.679189425150607 | Hp,Hc: (6, 3)
Sub_Episode: 6 | avg. reward: 17.967751967870054 | Hp,Hc: (6, 3)
Sub_Episode: 7 | avg. reward: 17.49443333077277 | Hp,Hc: (6, 3)
Sub_Episode: 8 | avg. reward: 16.918220520967107 | Hp,Hc: (6, 3)
Sub_Episode: 9 | avg. reward: 20.766785305660882 | Hp,Hc: (6, 3)
Sub_Episode: 10 | avg. reward: 17.531449653557612 | Hp,Hc: (6, 3)
Sub_Episode: 11 | avg. reward: 20.058981704067996 | Hp,Hc: (6, 3)
Sub_Episode: 12 | avg. reward: 17.32310239927987 | Hp,Hc: (6, 3)
Sub_Episode: 13 | avg. reward: 17.433026854103947 | Hp,Hc: (6, 3)
Sub_Episode: 14 | avg. reward: 18.160569870021124 | Hp,Hc: (6, 3)
Sub_Episode: 15 | avg. reward: 19.64864964828551 | Hp,Hc: (6, 3)
Sub_Episode: 16 | avg. 

## Plotting And Export

In [6]:
# Generate saved figures and any requested MPC comparison plots.
out_dir_rl = plot_horizon_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "recipe_counts": True,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
    },
)

out_dir_cmp = compare_mpc_rl_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=BASELINE_MPC_PATH,
    reward_fn=reward_fn,
    directory=RESULT_DIR,
    prefix_name=COMPARE_PREFIX,
    compare_mode=RUN_MODE,
    start_episode=COMPARE_START_EPISODE,
    save_pdf=SAVE_PDF,
    style_profile=STYLE_PROFILE,
)

print(out_dir_rl)
print(out_dir_cmp)

try:
    system.close(SNAPS_PATH)
except Exception:
    pass


C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Distillation\Results\distillation_dueling_horizon_disturb_fluctuation_standard_unified\20260420_173113
C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Distillation\Results\distillation_compare_dueling_horizon_disturb_fluctuation_standard\20260420_173121
